In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset, random_split, Dataset
import torchvision
from torch.utils.tensorboard import SummaryWriter

In [2]:
import torchvision
# For image transforms
from torchvision import transforms
# For DATA SET
import torchvision.datasets as datasets
# For Pytorch methods
import torch
import torch.nn as nn
# For Optimizer
import torch.optim as optim
# FOR DATA LOADER
from torch.utils.data import DataLoader
# FOR TENSOR BOARD VISUALIZATION
from torch.utils.tensorboard import SummaryWriter # to print to tensorboard

In [3]:
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
# Hyperparameters

lr = 3e-4
batchSize = 32  # Batch size
numEpochs = 100
logStep = 625  # the number of steps to log the images and losses to tensorboard

latent_dimension = 128 # 64, 128, 256

image_dimension = 28 * 28 * 1  # 784

# we define a tranform that converts the image to tensor and normalizes it with mean and std of 0.5
# which will convert the image range from [0, 1] to [-1, 1]
myTransforms = transforms.Compose([transforms.ToTensor(),transforms.Normalize((0.5,), (0.5,))])

In [5]:


# the MNIST dataset is available through torchvision.datasets
print("loading MNIST digits dataset")
dataset = datasets.MNIST(root="dataset/", transform=myTransforms, download=True)
# let's create a dataloader to load the data in batches
loader = DataLoader(dataset, batch_size=batchSize, shuffle=True)


loading MNIST digits dataset


In [6]:
class Generator(nn.Module):
    """
    Generator Model
    """
    def __init__(self, latent_dimension=latent_dimension, channels=1, features=64):
        super(Generator, self).__init__() 
        self.gen = nn.Sequential( 
            
            # Block 1: Input is the latent vector Z.
            # Shape: (latent_dim) x 1 x 1 -> Outputs: (features * 4) x 7 x 7
            # Note: A kernel size of 7 with no padding maps 1x1 exactly to 7x7
            nn.ConvTranspose2d(latent_dimension, features * 4, kernel_size=7, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(features * 4),
            nn.ReLU(True),

            # Block 2: Upsample by a factor of 2.
            # Shape: (features * 4) x 7 x 7 -> Outputs: (features * 2) x 14 x 14
            nn.ConvTranspose2d(features * 4, features * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features * 2),
            nn.ReLU(True),

            # Block 3: Final layer, upsample by 2 and output 1 channel.
            # Shape: (features * 2) x 14 x 14 -> Outputs: (channels=1) x 28 x 28
            nn.ConvTranspose2d(features * 2, channels, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh() 
        )
            

    def forward(self, x):
        return self.gen(x)

In [7]:

class Discriminator(nn.Module):
    """
    Discriminator Model
    """
    def __init__(self,channels=1, features=64):
        super(Discriminator, self).__init__()
        self.disc = nn.Sequential(
            # Block 1: Input is an image of shape (channels) x 28 x 28
            # Halves spatial dimensions to 14 x 14
            nn.Conv2d(channels, features, kernel_size=4, stride=2, padding=1, bias=False),
            # Standard practice: Use LeakyReLU in the discriminator
            nn.LeakyReLU(0.2, inplace=True),

            # Block 2: Downsample again
            # Halves spatial dimensions to 7 x 7
            nn.Conv2d(features, features * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # Block 3: Final classification layer
            # A 7x7 kernel with 0 padding maps the 7x7 spatial map exactly to 1x1
            nn.Conv2d(features * 2, 1, kernel_size=7, stride=1, padding=0, bias=False),
            nn.Sigmoid() # Outputs a probability (0 = fake, 1 = real)
        )

    def forward(self, x):
        return self.disc(x)

In [8]:
from torch.utils.tensorboard import SummaryWriter

# Initialize the writer and specify a directory for the logs
writer = SummaryWriter(log_dir="runs/GAN_graph_with_noise")

In [9]:
# Load the TensorBoard notebook extension
%load_ext tensorboard

# Start TensorBoard pointing to the directory where your writer is saving logs
%tensorboard --logdir runs --port 8008 --bind_all

Reusing TensorBoard on port 8008 (pid 17368), started 1:45:24 ago. (Use '!kill 17368' to kill it.)

In [ ]:
# initialize networks and optimizers
discriminator = Discriminator().to(device)
generator = Generator().to(device)
opt_discriminator = optim.Adam(discriminator.parameters(), lr=lr / 2.0, betas=(0.5, 0.999))
opt_generator = optim.Adam(generator.parameters(), lr=lr, betas=(0.5, 0.999))
#opt_discriminator = optim.Adam(discriminator.parameters(), lr=lr)
#opt_generator = optim.Adam(generator.parameters(), lr=lr)
fixed_noise = torch.randn(64, latent_dimension, 1, 1).to(device)
# This is a binary classification task, so we use Binary Cross Entropy Loss
criterion = nn.BCELoss()


# Training Loop
step = 0
# Step 1: Define a small noise scale
noise_factor = 0.1 

print("Started Training and visualization...")
for epoch in range(numEpochs):
    # loop over batches
    print()
    for batch_idx, (real, _) in enumerate(loader):
        # First we train the discriminator on real images vs. generated images

        # We do NOT flatten the image. We keep the 2D spatial dimensions for the CNN.
        # Shape remains (Batch, 1, 28, 28)
        real = real.to(device)
        batch_size = real.shape[0]

        # Step 1) generate fake images
        noise = torch.randn(batch_size, latent_dimension, 1, 1).to(device)
        fake = generator(noise)

        # Step 2) Train Discriminator:
        opt_discriminator.zero_grad()
        # - predict the discriminator output for real images
        # Step 2: Add noise to real images
        real_noisy = real + noise_factor * torch.randn_like(real)
        dis_pred_real = discriminator(real_noisy)
        # - real images are labeled as 1
        # - calculate the loss for real images
        #criterion_real = criterion(dis_pred_real, torch.ones_like(dis_pred_real))
        criterion_real = criterion(dis_pred_real, torch.ones_like(dis_pred_real) * 0.9)
        # - predict the discriminator output for fake images
        # Step 3: Add noise to fake images
        fake_noisy = fake.detach() + noise_factor * torch.randn_like(fake)
        dis_pred_fake = discriminator(fake_noisy.detach())

        
        # -fake images are labeled as 0
        # -calculate the loss for fake images
        criterion_fake = criterion(dis_pred_fake, torch.zeros_like(dis_pred_fake))
        # -average the loss for real and fake images
        loss_discriminator = (criterion_real + criterion_fake) / 2
        # - now upadate the weights of the discriminator by backpropagating the loss through the discriminator
        # the generator is not updated in this step
        # HINT: call the `backward` method of the discriminator with the argument `retain_graph=True` to keep the computational graph
        # this is necessary because we will use the same discriminator to train the generator
        loss_discriminator.backward()
        opt_discriminator.step()


        # Step 3) Train Generator:
        # Now train the generator by generating fake images and passing them through the discriminator
        opt_generator.zero_grad()
        dis_pred_fake_for_generator = discriminator(fake)
        loss_generator = criterion(dis_pred_fake_for_generator, torch.ones_like(dis_pred_fake_for_generator))
        loss_generator.backward()
        opt_generator.step()

        # You can do a little trick and modify the original objective function of
        # "minimizing the probability of the discriminator predicting the fake images as fake"
        # to "maximizing the probability of the discriminator predicting the fake images as real"
        # this leads to a faster training of the generator when it does not represent the real data well
        # this is a common trick in GANs
        # for more information see section 17.1.2 of the book Deep Learning by Bishop and Bishop



        # print the progress
        print(f"\rEpoch [{epoch}/{numEpochs}] Batch {batch_idx}/{len(loader)} | Loss discriminator: {loss_discriminator:.4f}, loss generator: {loss_generator:.4f}", end="")

        # Log the losses and example images to tensorboard
        if batch_idx % logStep == 0:
            with torch.no_grad():
                # Generate noise via Generator, we always use the same noise to see the progression
                fake = generator(fixed_noise).reshape(-1, 1, 28, 28)
                # Get real data
                data = real.reshape(-1, 1, 28, 28)
                # make grid of pictures and add to tensorboard
                imgGridFake = torchvision.utils.make_grid(fake, normalize=True)
                imgGridReal = torchvision.utils.make_grid(data, normalize=True)

                # TODO: add the images and losses to tensorboard
                # HINT: use the SummaryWriter to add the images and scalars to tensorboard
                # HINT: use the `add_image` method to add the images to tensorboard
                # HINT: use the `add_scalar` method to add the losses to tensorboard
                writer.add_image("Fake Images", imgGridFake, global_step=step)
                writer.add_image("Real Images", imgGridReal, global_step=step)
                writer.add_scalar("Loss/Discriminator", loss_discriminator.item(), global_step=step)
                writer.add_scalar("Loss/Generator", loss_generator.item(), global_step=step)

                # increment step
                step += 1

Started Training and visualization...

Epoch [0/100] Batch 1874/1875 | Loss discriminator: 0.3434, loss generator: 0.8282
Epoch [1/100] Batch 1874/1875 | Loss discriminator: 0.3836, loss generator: 0.7135
Epoch [2/100] Batch 399/1875 | Loss discriminator: 0.3850, loss generator: 0.9372